# المحاضرة 39: بناء مسار RAG متكامل مع التقطيع الدلالي المعياري
## القسم الأول: الشرح النظري والمعماري الشامل

### 1. الدوافع الهندسية والانتقال للبرمجة المعيارية (Modular Coding)
في المحاضرات السابقة تم تطبيق التقطيع الدلالي باستخدام دوال إجرائية منفصلة. في بيئات الإنتاج الحقيقية (Production)، تتطلب معمارية البرمجيات تجميع المنطق البرمجي داخل فئات (Classes) قابلة لإعادة الاستخدام والصيانة.

الهدف في هذه المرحلة:
- تغليف منطق حساب المتجهات والتشابه داخل كلاس موحد: `ThresholdSemanticChunker`.
- التوافق الكامل مع بنية وثائق لانج تشين (`langchain_core.documents.Document`).
- الحفاظ على البيانات الوصفية (`Metadata`) لكل وثيقة بعد التقطيع لضمان إمكانية تتبع المصدر أثناء الاسترجاع.

---

### 2. التصميم المعماري لكلاس `ThresholdSemanticChunker`
يتكون الكلاس من ثلاث مكونات رئيسية:
1. دالة البناء `__init__(model_name, threshold)`:
   - تحميل موديل التضمين المحلي (`all-MiniLM-L6-v2`).
   - ضبط عتبة التشابه الدلالي (افتراضياً 0.7 للقطع المحكم، أو 0.35 للقطع المتوازن).
2. دالة `split_text(text: str) -> List[str]`:
   - تجزئة النص إلى جمل مستقلة.
   - حساب المتجهات الكثيفة للجمل عبر `model.encode`.
   - قياس تشابه جيب التمام الزوجي: $\text{Sim}(E_{i-1}, E_i)$.
   - تجميع الجمل المتتابعة طالما $\text{Sim} \ge \text{Threshold}$، وقطع النص فور حدوث هبوط يقل عن العتبة.
3. دالة `split_documents(docs: List[Document]) -> List[Document]`:
   - الدوران على كل وثيقة، واستخراج محتواها النصي، وتطبيق `split_text`.
   - إنشاء كائنات `Document` جديدة لكل قطعة مع نسخ الـ `metadata` الخاصة بالوثيقة الأصلية.

---

### 3. مخطط تدفق البيانات في مسار RAG الكامل (LCEL Pipeline)

```mermaid
flowchart TD
    subgraph DataIngestion ["مرحلة التجهيز والتقطيع الدلالي"]
        A["مستند متعدد الموضوعات
(LangChain + برج إيفل وفرنسا)"] --> B["ThresholdSemanticChunker
(threshold = 0.7)"]
        B --> C["قطع وثائقية دلالية نقية
(4 Document Chunks)"]
    end

    subgraph VectorStorage ["مرحلة الفهرسة والاسترجاع"]
        C --> D["HuggingFaceEmbeddings
(all-MiniLM-L6-v2)"]
        D --> E["فهرس FAISS Vector Store"]
        E --> F["Retriever (k=2)"]
    end

    subgraph LCELInference ["مرحلة الاستدلال بتعبيرات LCEL"]
        Q["سؤال المستخدم:
'What is LangChain used for?'"] --> R["RunnableParallel
(context: retriever | question: passthrough)"]
        F --> R
        R --> P["ChatPromptTemplate
('Answer based on context...')"]
        P --> LLM["ChatGroq
(openai/gpt-oss-20b)"]
        LLM --> Out["StrOutputParser
(إجابة مركزة وخالية من الهلوسة)"]
    end
```

---

### 4. اعتبارات الإنتاج والـ Trade-offs
- **دقة الاسترجاع وعزل السياق**: يضمن التقطيع الدلالي عدم تلوث سياق الـ LLM بمعلومات غير ذات صلة (مثل معلومات باريس عند السؤال عن أدوات الذكاء الاصطناعي).
- **استهلاك الموارد**: يتم حساب التضمينات محلياً باستخدام موديل خفيف، مما يوفر تكاليف استدعاء الـ APIs الخارجية في مرحلة الـ Ingestion.


## القسم الثاني: التطبيق البرمجي المتكامل
يتضمن هذا القسم الكود البرمجي الكامل والنموذجي، موثقاً بالتعليقات التوضيحية وبدون أي تشتت.

In [1]:
# 1. استدعاء المكتبات وتهيئة متغيرات البيئة
import os
import warnings
from typing import List
from dotenv import load_dotenv

# Suppress minor deprecation warnings for cleaner output
warnings.filterwarnings("ignore")
load_dotenv()

# Machine learning and NLP libraries
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# LangChain core and community components
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# Verify environment configuration
has_groq = bool(os.getenv("GROQ_API_KEY"))
print(f"[CONFIG] GROQ_API_KEY configured: {has_groq}")

[CONFIG] GROQ_API_KEY configured: True


In [2]:
# 2. تعريف كلاس التقطيع الدلالي المعياري (ThresholdSemanticChunker)
class ThresholdSemanticChunker:
    """
    Modular semantic chunker class that splits documents based on
    consecutive sentence embedding similarity thresholds.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2", threshold: float = 0.7):
        """
        Initialize the chunker with an embedding model and a similarity threshold.
        """
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split_text(self, text: str) -> List[str]:
        """
        Split raw text into semantic chunks based on similarity drops.
        """
        # Split into non-empty stripped sentences
        sentences = [s.strip() for s in text.split("\n") if s.strip()]
        if not sentences:
            return []

        # Generate dense embeddings for all sentences
        embeddings = self.model.encode(sentences)

        chunks = []
        current_chunk = [sentences[0]]

        # Compare adjacent sentence embeddings
        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk = [sentences[i]]

        if current_chunk:
            chunks.append(" ".join(current_chunk))

        return chunks

    def split_documents(self, docs: List[Document]) -> List[Document]:
        """
        Split a list of LangChain Document objects while preserving metadata.
        """
        result = []
        for doc in docs:
            chunks = self.split_text(doc.page_content)
            for chunk in chunks:
                result.append(
                    Document(
                        page_content=chunk,
                        metadata=doc.metadata.copy()
                    )
                )
        return result

print("[INFO] ThresholdSemanticChunker class defined successfully.")

[INFO] ThresholdSemanticChunker class defined successfully.


In [3]:
# 3. إعداد المستند النموذجي وتشغيل التقطيع الدلالي
sample_text = """LangChain is a framework for building applications with LLMs.
LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

# Create input LangChain Document with metadata
raw_doc = Document(
    page_content=sample_text,
    metadata={"source": "sample_text.txt", "author": "Course Lecture 39"}
)

# Instantiate chunker with threshold = 0.7
chunker = ThresholdSemanticChunker(threshold=0.7)
doc_chunks = chunker.split_documents([raw_doc])

print(f"[INFO] Total generated Document chunks: {len(doc_chunks)}\n")
for idx, chunk in enumerate(doc_chunks, 1):
    print(f"--- Chunk [{idx}] | Source: {chunk.metadata.get('source')} ---")
    print(f"{chunk.page_content}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Total generated Document chunks: 4

--- Chunk [1] | Source: sample_text.txt ---
LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.

--- Chunk [2] | Source: sample_text.txt ---
You can create chains, agents, memory, and retrievers.

--- Chunk [3] | Source: sample_text.txt ---
The Eiffel Tower is located in Paris.

--- Chunk [4] | Source: sample_text.txt ---
France is a popular tourist destination.



In [4]:
# 4. فهرسة القطع في مخزن المتجهات FAISS وبناء المسترجع
# Initialize local embedding model for vector storage
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create FAISS vector store directly from semantic Document chunks
vector_store = FAISS.from_documents(doc_chunks, embeddings)

# Create retriever configured to fetch top 2 most relevant chunks
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Verify retriever functionality on test query
test_query = "What is LangChain used for?"
retrieved_docs = retriever.invoke(test_query)

print(f"[INFO] Retrieved {len(retrieved_docs)} chunks for query: '{test_query}':\n")
for i, d in enumerate(retrieved_docs, 1):
    print(f"[{i}] {d.page_content}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Retrieved 2 chunks for query: 'What is LangChain used for?':

[1] LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone.
[2] You can create chains, agents, memory, and retrievers.


In [5]:
# 5. بناء مسار LCEL الكامل وتنفيذ الاستعلام عبر Groq
# Define prompt template enforcing context adherence
prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context:
{context}

Question:
{question}

Answer:""")

# Initialize fast language model via Groq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.4)

# Assemble end-to-end LCEL pipeline
rag_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever,
        question=lambda x: x["question"]
    )
    | prompt
    | llm
    | StrOutputParser()
)

# Execute primary lecture query
query_1 = "What is LangChain used for?"
response_1 = rag_chain.invoke({"question": query_1})

print(f"[QUERY 1]: {query_1}")
print(f"[RESPONSE 1]:\n{response_1}\n")
print("-" * 60)

# Execute secondary query on geography to verify isolation
query_2 = "Where is the Eiffel Tower located?"
response_2 = rag_chain.invoke({"question": query_2})

print(f"[QUERY 2]: {query_2}")
print(f"[RESPONSE 2]:\n{response_2}")

[QUERY 1]: What is LangChain used for?
[RESPONSE 1]:
LangChain is a framework for building applications that use large language models (LLMs). It provides modular abstractions that let developers combine LLMs with tools such as OpenAI and Pinecone.

------------------------------------------------------------


[QUERY 2]: Where is the Eiffel Tower located?
[RESPONSE 2]:
Paris


In [6]:
# 6. التحقق البرمجي النهائي (Assertions)
assert len(doc_chunks) == 4, "Threshold 0.7 must yield exactly 4 semantic chunks"
assert "framework" in response_1.lower(), "Response 1 must identify LangChain as a framework"
assert "paris" in response_2.lower(), "Response 2 must correctly identify Paris"

print("[SUCCESS] All programmatic assertions passed successfully.")

[SUCCESS] All programmatic assertions passed successfully.


---
## القسم الثالث: مساحة التجربة والتطبيق العملي
استخدم الخلية التالية لتجربة نصوص جديدة، تعديل قيمة العتبة (مثل استخدام 0.35 لدمج كل موضوعات LangChain معاً)، أو طرح أسئلة جديدة على المسار.

In [7]:
# مساحة التجربة العملية:
# 1. جرب تغيير العتبة إلى 0.35 ولاحظ كيف تندمج الجمل المترابطة في قطعتين فقط
# 2. جرب إدخال نص مخصص من مجالك واختبر استرجاع الإجابات من المسار

custom_chunker = ThresholdSemanticChunker(threshold=0.35)
custom_chunks = custom_chunker.split_documents([raw_doc])

print(f"[PRACTICE] Chunks with balanced threshold (0.35): {len(custom_chunks)}\n")
for i, c in enumerate(custom_chunks, 1):
    print(f"[{i}] {c.page_content}\n")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[PRACTICE] Chunks with balanced threshold (0.35): 2

[1] LangChain is a framework for building applications with LLMs. LangChain provides modular abstraction to combine LLMs with tools like OpenAI and Pinecone. You can create chains, agents, memory, and retrievers.

[2] The Eiffel Tower is located in Paris. France is a popular tourist destination.

